# RTE teacher / baseline: DeBERTa-v3-base on Colab

Runs the **same code and pinned packages** as the local venv. Produces:
1. `outputs/baseline_deberta_v3_base/`: baseline metrics (5 seeds), dev predictions, best-val checkpoint
2. `outputs/baseline_deberta_v3_base_oof/teacher_logits_train.npy`: out-of-fold soft labels for distillation

Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100 is faster).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

name, memory.total [MiB]
Tesla T4, 15360 MiB
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


## 1. Get the code
Either clone your repo (set `REPO_URL`) or upload `rte-challenge.zip` (the project without `.venv/`, `data/`, `outputs/`).

In [ ]:
REPO_URL = "https://github.com/ANONYMIZED/rte-challenge.git"

%cd /content
if REPO_URL:
    !git clone -q $REPO_URL rte-challenge
else:
    from google.colab import files
    up = files.upload()  # choose rte-challenge.zip
    !unzip -q -o {next(iter(up))} -d /content
%cd /content/rte-challenge

/content
/content/rte-challenge


## 2. Install pinned dependencies (keep Colab's CUDA torch)
The universal lock is filtered so Colab's preinstalled CUDA build of torch (and its nvidia/triton libs) is untouched. If pip reports that numpy or another core package changed, use **Runtime → Restart session** and continue from the next cell.

In [ ]:
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!sed -i 's/<3.13/<3.14/' pyproject.toml
!pip install -q --no-deps -e .
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.7.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7

In [ ]:
!pip install -q -r /tmp/req-colab.txt

In [ ]:
!sed -i 's/<3.13/<3.14/' pyproject.toml

In [ ]:
!pip install -q --no-deps -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rte (pyproject.toml) ... done


## 3. Data
Upload the challenge zip (`Dataset for Challenge-20261101.zip`). The dev split is fetched from GLUE RTE validation.

In [ ]:
%cd /content/rte-challenge
from google.colab import files
up = files.upload()
ZIP = next(iter(up))
!python scripts/prepare_data.py --zip "$ZIP" --out data
!python -m pytest -q

/content/rte-challenge


Saving Dataset for Challenge-20261101.zip to Dataset for Challenge-20261101.zip
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (










train: 2490 rows, labels={'not_entailment': 1241, 'entailment': 1249}
dev: 277 rows, labels={'not_entailment': 131, 'entailment': 146}
train/dev pair overlap: 0
.........                                                                [100%]
=============================== warnings summary ===============================
../../usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63
  /usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
    from pandas.core import (

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html


## 4. Baseline: DeBERTa-v3-base, 5 seeds
Each seed early-stops on a premise-grouped 10% hold-out of train; dev is scored once per seed.

In [ ]:
!python -m rte.train --config configs/baseline_deberta_v3_base.yaml

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
run baseline_deberta_v3_base | model microsoft/deberta-v3-base | device cuda




/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(



[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias        

## 5. Teacher logits (out-of-fold, 5 folds grouped by premise)

In [ ]:
!python -m rte.train --config configs/baseline_deberta_v3_base.yaml --oof

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
fold 1/5: train 1992 held-out 498
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias 

## 6. Download artifacts
Unzip into the local project root; this restores `outputs/baseline_deberta_v3_base*/`. The checkpoint (~0.7 GB) is optional: drop `INCLUDE_MODEL` if you only need metrics and teacher logits.

In [ ]:
INCLUDE_MODEL = False
excl = "" if INCLUDE_MODEL else "-x 'outputs/baseline_deberta_v3_base/model/*'"
!zip -q -r teacher_artifacts.zip outputs/baseline_deberta_v3_base outputs/baseline_deberta_v3_base_oof {excl}
!du -h teacher_artifacts.zip
from google.colab import files
files.download("teacher_artifacts.zip")

88K	teacher_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>